# 06. Ensemble — 複数モデルのブレンド

README の工程 **⑥ Ensemble**。本番スクリプトは `src/06_ensemble_hillclimb.py`。
このノートブックは同じ処理を分解して、**なぜこの構成になったか**を追えるようにしたもの。

## ここまでの積み上げ

| 段階 | CV | Public LB |
|---|---|---|
| ベースライン(CatBoost単体) | 0.94156 | 0.94170 |
| FE(厳密値TE ほか) | 0.94589 | — |
| HPO(列サブサンプリング) | 0.94610 | — |
| **アンサンブル** | **0.94623** | **0.94645** |

単体の最良(XGBoost 0.94608)に対し、ブレンドで **+0.00015**。
数字は小さいが、**2026-09-23 時点で上位15%のラインが LB 0.94641、上位10%が 0.94647** と
0.0001 未満の刻みで順位が動く世界なので、十分に意味のある差になる。

## なぜブレンドが効くのか

モデルごとに間違え方が違うため、平均すると誤差が打ち消し合う。
したがって**単体の強さより「間違え方が違うこと(非相関性)」が重要**になる。

実際このコンペでも、単体で最弱(0.94462)だった LightGBM のバリアントが、
非相関だったために重み 25% で採用された局面があった(**diversity beats strength**)。

In [ ]:
import os, sys
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.path.abspath("src"))
print("cwd:", os.getcwd())

In [ ]:
import glob, importlib, itertools
import numpy as np
import pandas as pd
from scipy.stats import rankdata, spearmanr
from sklearn.metrics import roc_auc_score

train = pd.read_csv("data/train.csv")
test = pd.read_csv("data/test.csv")
y = (train["Will_Buy_EV"] == "Yes").astype(int).values
print(f"train {len(train):,} / test {len(test):,}")

## 候補の読み込み

`oof/` にある **OOF 予測(学習に使わなかった行への予測)** と、対応する test 予測を全部拾う。

各モデルは fold 分割を `StratifiedKFold(5, shuffle=True, random_state=42)` で**完全に揃えて**いる。
これが揃っていないと OOF 同士を足すことも、後の DeLong 検定もできない。

### なぜ rank に変換するのか

モデルごとに出力する確率のスケールが違う(あるモデルは 0.2 前後に集中、別のモデルは広く散る)。
そのまま平均すると散らばりの大きいモデルの影響が強く出てしまう。
**順位に変換してから平均**すれば、各モデルを対等に扱える。AUC は順位だけで決まる指標なので、
この変換で情報は失われない。

In [ ]:
cands = {}
for f in sorted(glob.glob("oof/oof_*.npy")):
    name = os.path.basename(f).replace("oof_", "").replace(".npy", "")
    pred_path = f"oof/pred_{name}.npy"
    if os.path.exists(pred_path):
        cands[name] = (rankdata(np.load(f)) / len(y),          # OOF(順位に変換)
                       rankdata(np.load(pred_path)) / len(test))  # test(同上)

tbl = pd.DataFrame(
    [(n, roc_auc_score(y, o)) for n, (o, _) in cands.items()],
    columns=["候補", "OOF AUC"]
).sort_values("OOF AUC", ascending=False).reset_index(drop=True)
print(f"候補 {len(cands)} 本")
tbl.head(10)

## 主力4モデルの相関

**ここが最も重要な図。** 相関が高いほど「同じ間違え方」をしており、足しても伸びない。

In [ ]:
main = [m for m in ["lgbm", "xgb", "catboost", "realmlp"] if m in cands]
rows = []
for a, b in itertools.combinations(main, 2):
    rows.append([a, b, spearmanr(cands[a][0], cands[b][0]).statistic])
corr = pd.DataFrame(rows, columns=["A", "B", "順位相関"]).sort_values("順位相関")
corr

GBDT 同士(LightGBM / XGBoost / CatBoost)は **0.993〜0.999** と極めて似ている。
特に LightGBM × XGBoost は、両方に同じ「列サブサンプリング + 深さ5」を入れた結果 **0.9986** まで同質化した。

一方 **RealMLP だけが 0.994 前後と低い**。ニューラルネットで仕組みが根本的に違うため、
ここが多様性の供給源になっている。現在の重み 1/3 はこの非相関性によるもので、
単体スコア(0.94589、4モデル中3位)で選ばれているわけではない。

## hill climbing(貪欲法)

空の状態から始めて、**「今の平均に足すと最も良くなる候補」を1本ずつ選ぶ**だけの単純な方法。
同じ候補を複数回選べるようにしてあり、**選ばれた回数がそのまま重み**になる。

Nelder-Mead などで重みを連続的に最適化する方法もあるが、候補が20本以上あると
過学習しやすい。貪欲法は選択肢が離散的なぶん、過学習しにくい。

In [ ]:
selected, cur_sum, best_auc = [], np.zeros(len(y)), 0.0
names = list(cands)

for step in range(15):
    scores = {n: roc_auc_score(y, (cur_sum + cands[n][0]) / (len(selected) + 1)) for n in names}
    pick = max(scores, key=scores.get)
    if scores[pick] <= best_auc + 1e-7:   # 改善が止まったら終了
        break
    best_auc = scores[pick]
    selected.append(pick)
    cur_sum = cur_sum + cands[pick][0]
    print(f"step {step+1}: +{pick:20s} -> {best_auc:.5f}")

weights = {n: selected.count(n) / len(selected) for n in set(selected)}
print(f"\n最終 OOF AUC: {best_auc:.5f}")
print("重み:", {k: round(v, 4) for k, v in sorted(weights.items(), key=lambda kv: -kv[1])})

## ⚠ ここで終わらせない — 工程⑦(DeLong 検定)が必要な理由

**貪欲法は OOF 上の偶然を拾う。** 候補を20本以上並べて「一番良くなるもの」を選び続ければ、
たまたま OOF に合っただけの組み合わせが選ばれうる。

実際にそれで失敗した記録がある。

| 構成 | CV | z値 | Public LB |
|---|---|---|---|
| 3モデル | 0.946234 | — | **0.94645** |
| 5点(貪欲法が「良い」と判断) | 0.946243(**+0.000009**) | +1.57(有意でない) | **0.94643**(-0.00002) |

CV では良くなったのに、**LB では逆に下がった**。DeLong 検定は事前に「有意でない」と判定していた。

`src/07_compare_oof.py` はこの判定を行う。**スコアを作るスクリプトではなく、
採用するかどうかを決める「関門」**である。

In [ ]:
cmp = importlib.import_module("07_compare_oof")

ens = cur_sum / len(selected)                       # 今回のアンサンブル
best_single = max(main, key=lambda m: roc_auc_score(y, cands[m][0]))

a0, a1, diff, se, z, p = cmp.paired_test(y, cands[best_single][0], ens)
print(f"単体最良 ({best_single}): {a0:.6f}")
print(f"アンサンブル          : {a1:.6f}")
print(f"差分 {diff:+.6f}  SE {se:.6f}  z {z:+.2f}  p {p:.2e}")
print("→ 採用" if (diff >= 8e-5 and z >= 3) else "→ 誤差(採用しない)")

## 提出ファイルの作成

OOF で決めた重みを、そのまま test 予測に適用する。

In [ ]:
test_pred = sum(w * cands[n][1] for n, w in weights.items())
sub = pd.DataFrame({"id": test["id"], "Will_Buy_EV": test_pred})

# 本番の submit/submission_hillclimb.csv を上書きしないよう別名で保存する
sub.to_csv("submit/submission_from_notebook.csv", index=False)
print(sub.shape)
sub.head()

## 提出履歴と教訓

| 提出 | 構成 | CV | Public LB | 順位相当 |
|---|---|---|---|---|
| CatBoost 単体 | — | 0.94156 | 0.94170 | 1637位 |
| 4モデル | RealMLP .4 / CatBoost .2 / LGBM .2 / XGB .2 | 0.94618 | 0.94636 | 523位 |
| **3モデル** | **LGBM 1/3 / XGB 1/3 / RealMLP 1/3** | **0.94623** | **0.94645** | **293位** |
| 5点 | 上記 + 2バリアント | 0.946243 | 0.94643 | (下がった) |

> 順位は **2026-09-23 時点の暫定値**(2,732チーム)の LB で、そのスコアが今何位に相当するかを
> 示したもの。提出当時の順位ではない。**実際の現在順位は 320位**(0.94645 に 32チームが並んでおり、
> Kaggle は同点を提出時刻順に並べるため)。

**得られた教訓**

1. **単体を強くすることと多様性はトレードオフ。** 4モデル全てを同じ最強構成に揃えたとき、
   同質化してアンサンブルの上積みが +0.00002 まで縮んだ
2. **弱くても非相関なら採用される。** 単体最弱のバリアントが重み25%を得た局面があった
3. **CV の非有意な改善は LB で再現しない。** 必ず DeLong 検定を通してから提出する
4. CatBoost は単体では健闘していた(0.94589)が、**他のGBDTと同質だったため最終的に重み0**になった